In [13]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy
import json
from sklearn.model_selection import GroupShuffleSplit

In [14]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_dict_imu.pkl', 'rb') as f: 
    imu_dict = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_minze_dict.pkl', 'rb') as f:
    ground_truth_dict = pickle.load(f)
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/data/df_dict_urineestimate_method1.pkl', 'rb') as f:
    urine_estimate_dict = pickle.load(f)

### Step 1
Remove the data with the almost the entire data being void.
1. subj_9_void4
2. subj_11_void2

In [15]:
del imu_dict['subj_9_void4']
del imu_dict['subj_11_void2']

del ground_truth_dict['subj_9_void4']
del ground_truth_dict['subj_11_void2']

### Step 2

1. Add experiment ids to all experiments.
2. Combine the all experiments into one Dataframe.

In [16]:

imu_df_with_ids = pd.DataFrame()
dict = copy.deepcopy(imu_dict)
ground_truth_dict_with_ids = pd.DataFrame()
gt_dict = copy.deepcopy(ground_truth_dict)

for exp_id, void_instance in tqdm(enumerate(dict.keys()), desc="Adding experiment ids to IMU data"):
    acc = dict[void_instance]
    gt = gt_dict[void_instance]
        
    # Add the experiment id
    acc['experiment_id'] = exp_id + 1
    gt['experiment_id'] = exp_id + 1
    
    # appeding to the DataFrame
    imu_df_with_ids = pd.concat([imu_df_with_ids, acc], ignore_index=True)
    ground_truth_dict_with_ids = pd.concat([ground_truth_dict_with_ids, gt], ignore_index=True)

Adding experiment ids to IMU data: 0it [00:00, ?it/s]

Adding experiment ids to IMU data: 41it [00:00, 457.22it/s]


In [17]:
imu_df_with_ids

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,time,Real time,experiment_id
0,0.066242,-13.923373,-17.075456,-38.322404,8.725663,83.141039,0.000000,1900-01-01 00:00:00.000000,1
1,0.042000,-13.944164,-17.017063,-26.325645,2.767256,81.080344,0.016935,1900-01-01 00:00:00.016935,1
2,0.006946,-13.985796,-16.976153,-12.871816,-2.367739,91.281087,0.033871,1900-01-01 00:00:00.033871,1
3,-0.012306,-14.016341,-16.957208,-4.652553,-4.760758,130.487448,0.050806,1900-01-01 00:00:00.050806,1
4,0.014324,-13.933079,-16.921674,3.602771,-1.295901,187.665522,0.067741,1900-01-01 00:00:00.067741,1
...,...,...,...,...,...,...,...,...,...
144265,-0.415704,-9.772312,-5.398604,-5.791146,-39.759699,5.077117,41.110177,1900-01-01 00:00:41.110177,41
144266,-0.447311,-9.756686,-5.420740,-14.286152,-42.545283,-8.623367,41.126721,1900-01-01 00:00:41.126721,41
144267,-0.432375,-9.771661,-5.388947,-23.709813,-46.282834,-24.237577,41.143264,1900-01-01 00:00:41.143264,41
144268,-0.405955,-9.756496,-5.326923,-19.426838,-51.825114,-38.346962,41.159807,1900-01-01 00:00:41.159807,41


## Step 3
1. Split data into test and train based on experiment id.

In [ ]:
#'groups' is the df['experiment_id'] column

# --- Step 1: Split your data by experiment ID ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(imu_df_with_ids, groups=imu_df_with_ids['experiment_id']))

train_df = imu_df_with_ids.iloc[train_idx]
test_df = imu_df_with_ids.iloc[test_idx]
groups_train = train_df['experiment_id'].unique()
groups_test = test_df['experiment_id'].unique()





Normalization parameters saved!
{'means': {'acc_x': 0.149932295509798, 'acc_y': -12.752894996422528, 'acc_z': -13.041020370498591, 'gyr_x': -1.0776177064910557, 'gyr_y': -17.937525803392965, 'gyr_z': 9.982370340444326}, 'stds': {'acc_x': 0.7779419274245848, 'acc_y': 1.685952615534354, 'acc_z': 4.229702303342251, 'gyr_x': 97.85956118863069, 'gyr_y': 196.78995085049831, 'gyr_z': 597.4237994865183}}


## Step 4
1. Calculate the normalization parameters based on the train set
2. Normalize the data and pickle the dataframes

In [ ]:
cols_to_normalize = ['acc_x', 'acc_y', 'acc_z', 'gyr_x', 'gyr_y', 'gyr_z']

# Calculate means and stds from the combined training data
train_means = train_df[cols_to_normalize].mean()
train_stds = train_df[cols_to_normalize].std()

# Store parameters in a dictionary
norm_params = {
    'means': train_means.to_dict(),
    'stds': train_stds.to_dict()
}

# Save the dictionary to a JSON file
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/normalization_parameters.json', 'w') as f:
    json.dump(norm_params, f, indent=4)

print("Normalization parameters saved!")
print(norm_params)

def apply_normalization(df_to_norm: pd.DataFrame, params: dict) -> pd.DataFrame:
    means = params['means']
    stds = params['stds']
    
    for col in cols_to_normalize:
        df_to_norm[col] = (df_to_norm[col] - means[col]) / stds[col]
        
    return df_to_norm

# Apply the same transformation to both train and test sets
train_df_normalized = apply_normalization(train_df.copy(), norm_params)
test_df_normalized = apply_normalization(test_df.copy(), norm_params)



In [20]:
train_df_normalized

,acc_x,acc_y,acc_z,gyr_x,gyr_y,gyr_z,time,Real time,experiment_id
0,-0.107579,-0.694253,-0.953834,-0.380594,0.135491,0.122457,0.000000,1900-01-01 00:00:00.000000,1
1,-0.138740,-0.706585,-0.940029,-0.258003,0.105213,0.119008,0.016935,1900-01-01 00:00:00.016935,1
2,-0.183801,-0.731279,-0.930357,-0.120522,0.079119,0.136082,0.033871,1900-01-01 00:00:00.033871,1
3,-0.208548,-0.749396,-0.925878,-0.036531,0.066959,0.201708,0.050806,1900-01-01 00:00:00.050806,1
4,-0.174317,-0.700010,-0.917477,0.047828,0.084565,0.297416,0.067741,1900-01-01 00:00:00.067741,1
...,...,...,...,...,...,...,...,...,...
130165,-3.973329,0.064331,0.181787,-0.237990,-1.063046,-2.498653,37.138237,1900-01-01 00:00:37.138237,39
130166,-4.053467,-0.101806,0.210093,-0.119794,-0.969203,-2.670028,37.154779,1900-01-01 00:00:37.154779,39
130167,-3.620434,-0.230684,-0.009250,-0.805102,-0.593700,-2.681534,37.171322,1900-01-01 00:00:37.171322,39
130168,-3.177720,-0.213060,-0.122035,-0.957332,-0.434202,-2.738270,37.187864,1900-01-01 00:00:37.187864,39


In [21]:
# # pickle the DataFrames
# with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/train_df_normalized.pkl', 'wb') as f:
#     pickle.dump(train_df_normalized, f) 
# with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/test_df_normalized.pkl', 'wb') as f:
#     pickle.dump(test_df_normalized, f)  

# with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/deployable/data/ground_truth_with_ids.pkl', 'wb') as f:
#     pickle.dump(ground_truth_dict_with_ids, f)